# 02 · Bidirectional Mamba SSM

Part of the study *Multi-Label 12-Lead ECG Classification on the PTB-XL Dataset* ([arXiv:2609.12803](https://arxiv.org/abs/2609.12803)). Run order, folder layout, and results: [project README](../README.md).

> GPU: A100 (L4/T4 acceptable). `batch_size=32`.

Shared protocol (`ecg_common.py`): official PTB-XL folds 1–8 / 9 / 10, checkpoint = validation macro-AUPRC, val F1-max thresholds, reported metrics = test fold 10. Outputs: `experiments/<model>/seed_<seed>/` and `results/<canonical>/seed_<seed>/`.
Reads `processed/sr100` from notebook 00.


In [ ]:
# COLAB_INSTALL
# Colab: mamba-ssm from GitHub wheels (cu12 + torch 2.10; cp310–cp313).
# Never pip-install mamba-ssm from generic PyPI: that silently compiles CUDA on T4.
import subprocess
import sys

TARGET_TORCH = '2.10.0'
TORCH_INDEX = 'https://download.pytorch.org/whl/cu128'
CAUSAL_TAG = 'v1.7.0'
CAUSAL_VER = '1.7.0'
MAMBA_TAG = 'v2.3.2.post1'
MAMBA_VER = '2.3.2.post1'
WHEEL_PY_TAGS = ('cp310', 'cp311', 'cp312', 'cp313')


def _pip(*args: str) -> None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *args])


def _pip_uninstall(*pkgs: str) -> None:
    subprocess.call(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', *pkgs],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )


def _smoke_test() -> None:
    for mod in list(sys.modules):
        if (
            mod == 'mamba_ssm'
            or mod.startswith('mamba_ssm.')
            or mod == 'causal_conv1d'
            or mod.startswith('causal_conv1d.')
        ):
            del sys.modules[mod]
    from mamba_ssm import Mamba

    d_model = 64
    layer = Mamba(d_model=d_model, d_state=16, d_conv=4, expand=2).cuda().eval()
    x = torch.randn(2, 128, d_model, device='cuda')
    with torch.no_grad():
        y = layer(x)
    assert y.shape == x.shape
    print('mamba OK |', 'out', tuple(y.shape), '|', torch.cuda.get_device_name(0))
    del layer, x, y
    torch.cuda.empty_cache()


_pip(
    '-q',
    'openpyxl',
    'scikit-learn',
    'pandas',
    'matplotlib',
    'tqdm',
    'einops',
    'ninja',
    'packaging',
)

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU runtime required. Colab: Runtime → Change runtime type → GPU, then re-run this cell.'
    )

py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
print(
    'torch:', torch.__version__,
    '| cuda:', torch.version.cuda,
    '| gpu:', torch.cuda.get_device_name(0),
    '| python:', py_tag,
)

if py_tag not in WHEEL_PY_TAGS:
    raise RuntimeError(
        f'No prebuilt mamba wheel for {py_tag}. '
        'Colab: Runtime → Change runtime type → Runtime version → 2026.04 '
        '(Python 3.12 + torch 2.10), then re-run this cell. '
        'Do not compile from source on T4.'
    )

torch_ver = torch.__version__.split('+')[0]
if not torch_ver.startswith('2.10.'):
    print(f'Pinning torch=={TARGET_TORCH}+cu128 for matching mamba wheels...')
    _pip(
        f'torch=={TARGET_TORCH}',
        'torchvision==0.25.0',
        'torchaudio==2.10.0',
        '--index-url',
        TORCH_INDEX,
    )
    print(
        '\n*** Torch reinstalled. Runtime → Restart session, then re-run THIS install cell. ***\n'
    )
    raise SystemExit('Restart Colab runtime, then re-run the install cell.')

wheel_abi = 'TRUE'
causal_whl = (
    f'https://github.com/Dao-AILab/causal-conv1d/releases/download/{CAUSAL_TAG}/'
    f'causal_conv1d-{CAUSAL_VER}+cu12torch2.10cxx11abi{wheel_abi}-{py_tag}-{py_tag}-linux_x86_64.whl'
)
mamba_whl = (
    f'https://github.com/state-spaces/mamba/releases/download/{MAMBA_TAG}/'
    f'mamba_ssm-{MAMBA_VER}+cu12torch2.10cxx11abi{wheel_abi}-{py_tag}-{py_tag}-linux_x86_64.whl'
)
_pip_uninstall('mamba-ssm', 'mamba_ssm', 'causal-conv1d', 'causal_conv1d')
print('Installing wheel (~1-2 min, not a compile)', causal_whl)
_pip(causal_whl)
print('Installing wheel (~500 MB, 1-2 min)', mamba_whl)
_pip(mamba_whl)

_smoke_test()


In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount('/content/drive')

SAMPLING_RATE = 100
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'ECG AI'
PROCESSED_DIR = PROJECT_DIR / 'processed' / f'sr{SAMPLING_RATE}'
EXPERIMENTS_DIR = PROJECT_DIR / 'experiments'
RESULTS_DIR = PROJECT_DIR / 'results'
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import load_processed_manifest

_manifest = load_processed_manifest(PROCESSED_DIR)
print('processed:', PROCESSED_DIR)
print('dataset_version:', _manifest.get('dataset_version') or _manifest.get('raw_root_name'))
print('split:', _manifest.get('split_by'), _manifest.get('train_folds'), _manifest.get('val_folds'), _manifest.get('test_folds'))


## Shared protocol

`notebooks/ecg_common.py` is imported. Checkpoint / early stop: **validation macro-AUPRC**. Architecture stays in this notebook; splits, metrics, thresholds, and the Trainer live in the shared module.


In [ ]:
from __future__ import annotations

import sys

import numpy as np
import pandas as pd
import torch
from torch import nn

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import (
    CLASSES,
    EARLY_STOPPING_PATIENCE,
    EVALUATE_TEST as DEFAULT_EVALUATE_TEST,
    ExperimentConfig,
    MAX_EPOCHS,
    PRIMARY_MONITOR,
    SEEDS,
    build_trainer,
    finalize_run_config,
)

EVALUATE_TEST = DEFAULT_EVALUATE_TEST  # True: score test fold 10 with val-tuned thresholds
ACTIVE_SEEDS = [42]  # paper: ACTIVE_SEEDS = SEEDS
print('monitor=', PRIMARY_MONITOR, '| epochs=', MAX_EPOCHS, '| patience=', EARLY_STOPPING_PATIENCE)
print('ACTIVE_SEEDS', ACTIVE_SEEDS, '| EVALUATE_TEST', EVALUATE_TEST)


## Model

Keep architecture hyperparameters (`d_model`, `depth`, `dropout`, …) in the `model_factory` below **and** in `experiment.extra` together; changing only one side does not update the model.


In [ ]:
# MODEL_ARCHITECTURE
try:
    from mamba_ssm import Mamba
except ImportError as exc:
    raise ImportError(
        'mamba-ssm missing. Run the first install cell on a GPU Colab runtime '
        '(torch 2.10 + prebuilt wheels), restart if prompted, then re-run install.'
    ) from exc


class BidirectionalMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int) -> None:
        super().__init__()
        self.forward_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.backward_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        forward_out = self.forward_mamba(x)
        backward_out = torch.flip(self.backward_mamba(torch.flip(x, dims=[1])), dims=[1])
        return self.norm(x + 0.5 * (forward_out + backward_out))


class BidirectionalMambaClassifier(nn.Module):
    def __init__(
        self,
        in_channels=12,
        num_classes=5,
        d_model=128,
        d_state=16,
        d_conv=4,
        expand=2,
        depth=4,
        dropout=0.2,
    ) -> None:
        super().__init__()
        self.input_proj = nn.Conv1d(in_channels, d_model, kernel_size=5, padding=2)
        self.blocks = nn.ModuleList(
            [BidirectionalMambaBlock(d_model, d_state, d_conv, expand) for _ in range(depth)]
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x).transpose(1, 2)
        for block in self.blocks:
            x = block(x)
        x = x.mean(dim=1)
        x = self.dropout(x)
        return self.classifier(x)


## Training hyperparameters

Hyperparameters are defined in `ExperimentConfig(...)` in the next cell.

Bidirectional Mamba: `batch_size=32`, `learning_rate=5e-4`. Shared protocol: `MAX_EPOCHS=50`, `EARLY_STOPPING_PATIENCE=7`, checkpoint = validation **macro-AUPRC**. `ACTIVE_SEEDS = [42]` is a single seed; for the paper use `ACTIVE_SEEDS = SEEDS` (`[42, 52, 62]`).
Checkpoints and thresholds are fit on val fold 9; reported performance analysis uses test fold 10.


In [ ]:
experiment = ExperimentConfig(
    model_name='bidirectional_mamba',
    processed_dir=str(PROCESSED_DIR),
    output_root=str(EXPERIMENTS_DIR),
    learning_rate=5e-4,
    weight_decay=1e-4,
    epochs=MAX_EPOCHS,
    batch_size=32,
    sampling_rate=SAMPLING_RATE,
    seed=42,
    patience=EARLY_STOPPING_PATIENCE,
    monitor=PRIMARY_MONITOR,
    evaluate_test=EVALUATE_TEST,
    results_root=str(RESULTS_DIR),
    extra={'d_model': 128, 'depth': 4, 'dropout': 0.2},
)

def model_factory(num_classes: int):
    return BidirectionalMambaClassifier(
        in_channels=12,
        num_classes=num_classes,
        d_model=128,
        d_state=16,
        d_conv=4,
        expand=2,
        depth=4,
        dropout=0.2,
    )

print('config:', experiment.model_name, '| lr', experiment.learning_rate, '| batch', experiment.batch_size, '| epochs', experiment.epochs)
print('X_train', tuple(np.load(Path(experiment.processed_dir) / 'X_train.npy', mmap_mode='r').shape))


In [ ]:
for seed in ACTIVE_SEEDS:
    experiment.seed = int(seed)
    trainer, data_module = build_trainer(experiment, model_factory=model_factory)
    history_df = trainer.fit(
        data_module.train_dataloader(),
        data_module.val_dataloader(),
        epochs=experiment.epochs,
    )
    protocol = trainer.evaluate_with_protocol(
        data_module.val_dataloader(),
        test_loader=data_module.test_dataloader(),
        evaluate_test=EVALUATE_TEST,
        results_root=RESULTS_DIR,
    )
    finalize_run_config(
        trainer,
        extra={'dataset_version': data_module.manifest.get('dataset_version') or data_module.manifest.get('raw_root_name')},
    )
    print('best val macro-AUPRC', protocol.get('best_validation_macro_auprc'), 'epoch', protocol.get('best_epoch'))

history_df.tail()


## Resume from last.pt

If the `fit` cell above was interrupted (Colab disconnect, OOM, runtime reset), run this cell. It loads `last.pt` (weights + optimizer) and writes remaining epochs into the same run folder.

After a kernel restart, skip the training cell: re-run the install, Drive, protocol, model, and `ExperimentConfig` cells, then this cell.
If `trainer` is missing, this cell rebuilds it with `build_trainer`. Do not point it at another model's `last.pt`.


In [ ]:
import json

RESUME_RUN_DIR = None  # None: experiments/bidirectional_mamba/seed_<seed>

if 'trainer' not in globals() or 'data_module' not in globals():
    trainer, data_module = build_trainer(experiment, model_factory=model_factory)

_resume_dir = Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else (
    Path(experiment.output_root) / experiment.model_name / f'seed_{int(experiment.seed)}'
)
_cfg_path = _resume_dir / 'config.json'
if _cfg_path.exists():
    _cfg = json.loads(_cfg_path.read_text(encoding='utf-8'))
    saved = str(_cfg.get('model_name') or '')
    if saved and saved != experiment.model_name:
        raise ValueError(
            f'Refuse resume: {_resume_dir} belongs to {saved}, this notebook is {experiment.model_name}.'
        )
if not (_resume_dir / 'checkpoints' / 'last.pt').exists():
    raise FileNotFoundError(f'last.pt not found: {_resume_dir / "checkpoints" / "last.pt"}')

history_df = trainer.fit(
    data_module.train_dataloader(),
    data_module.val_dataloader(),
    epochs=experiment.epochs,
    resume_from_last=True,
    resume_run_dir=str(_resume_dir),
)
protocol = trainer.evaluate_with_protocol(
    data_module.val_dataloader(),
    test_loader=data_module.test_dataloader(),
    evaluate_test=EVALUATE_TEST,
    results_root=RESULTS_DIR,
)
history_df.tail()


## Performance analysis

This section is **standalone**: after a runtime restart it can be run on its own, without the install, import, or training cells above.

Loads `experiments/<model>/seed_42/checkpoints/best.pt`. Thresholds are fit on val fold 9 (F1-max). Reported metrics, 2×2 confusion matrices, and prediction CSVs use **test fold 10**.
Files: `experiments/.../predictions/val_predictions.csv` and `test_predictions.csv`; copy under `results/<canonical>/seed_42/`.


In [ ]:
# ANALYZE_FROM_BEST_PT
# This cell runs standalone. Cells above (pip / imports / training) are not required.
import sys
import subprocess
from pathlib import Path

subprocess.check_call(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'openpyxl',
        'scikit-learn',
        'pandas',
        'numpy',
        'matplotlib',
        'seaborn',
        'tqdm',
    ]
)
try:
    from google.colab import drive

    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive') / 'ECG AI'
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from ecg_common import (
    evaluate_and_display_best_pt,
    exec_notebook_cells,
    extra_from_run_config,
    load_json,
    seed_run_dir,
)

EVALUATE_TEST = True  # always score test fold 10 from best.pt
ANALYSIS_SEED = 42
MODEL_NAME = 'bidirectional_mamba'
NOTEBOOK_PATH = PROJECT_DIR / 'notebooks' / '02_bidirectional_mamba_ssm.ipynb'
run_dir = seed_run_dir(PROJECT_DIR / 'experiments', MODEL_NAME, ANALYSIS_SEED)
ckpt = run_dir / 'checkpoints' / 'best.pt'
if not ckpt.exists():
    raise FileNotFoundError(f'best.pt not found: {ckpt}')

cfg = load_json(run_dir / 'config.json') if (run_dir / 'config.json').exists() else {}
extra = extra_from_run_config(cfg)

try:
    from mamba_ssm import Mamba  # noqa: F401
except ImportError:
    exec_notebook_cells(NOTEBOOK_PATH, '# COLAB_INSTALL', globals())

import torch
from torch import nn

exec_notebook_cells(NOTEBOOK_PATH, '# MODEL_ARCHITECTURE', globals())

def model_factory(num_classes: int):
    return BidirectionalMambaClassifier(
        in_channels=12,
        num_classes=num_classes,
        d_model=int(extra.get('d_model', 128)),
        d_state=int(extra.get('d_state', 16)),
        d_conv=int(extra.get('d_conv', 4)),
        expand=int(extra.get('expand', 2)),
        depth=int(extra.get('depth', 4)),
        dropout=float(extra.get('dropout', 0.2)),
    )

evaluate_and_display_best_pt(
    model_factory,
    run_dir,
    processed_dir=PROJECT_DIR / 'processed' / 'sr100',
    results_root=PROJECT_DIR / 'results',
    evaluate_test=True,
)
